In [51]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder

In [52]:
alarmes = pd.read_csv('../../data/Alarmes/alarmes_trata_limpos_consolidado.csv')
zm01 = pd.read_excel('../../data/Corretiva Emergencial/processados/ZM01_limpa.xlsx', sheet_name='ZM01_limpa')
zm02 = pd.read_excel('../../data/Corretiva Programada/processados/ZM02_limpa.xlsx', sheet_name='ZM02_limpa')
zm03 = pd.read_csv('../../data/Preventiva/preventiva_zm03_limpa.csv')

print(alarmes.shape, zm01.shape, zm02.shape, zm03.shape)

(742506, 15) (10543, 27) (1163, 27) (569, 19)


In [53]:
alarmes.head(5)

numeroAlarmesEstacao = alarmes['Estação'].value_counts()

## Pré Processamento

### Transformando em Datetime

In [96]:
zm01["T"] = pd.to_datetime(zm01["inicio_avaria_datetime"], errors="coerce")
zm01['fim_avaria_datetime'] = pd.to_datetime(zm01['fim_avaria_datetime'], errors='coerce')
alarmes['Horário do Alarme'] = pd.to_datetime(alarmes['Horário do Alarme'], errors='coerce')
alarmes['dt_alarme'] = pd.to_datetime(alarmes['dt_alarme'], errors='coerce')
alarmes['dt_normalizacao'] = pd.to_datetime(alarmes['dt_normalizacao'], errors='coerce')
alarmes.head(5)

,TAG,Horário do Alarme,Estação,Porta,Alarme,Horário da Normalizacao,Tempo de Normalização,tempo_normalizacao_seg,dt_alarme,dt_normalizacao,duracao_calculada_seg,estacao_extraida,plataforma_extraida,porta_extraida,tipo_evento_extraido
0,PSD_BUT_BUT1_PDM18_OBS,2023-10-17 21:44:30.900,BUT,18.0,OBS,2023-10-23 01:17:58.893,"5 dias, 03:33:28",444808,2023-10-17 21:44:30.900,2023-10-23 01:17:58.893,444807.993,BUT,BUT1,18.0,OBS
1,PSD_BUT_BUT1_PDM18_OBS,2023-10-17 21:44:30.900,BUT,18.0,OBS,2023-10-21 14:14:04.483,"3 dias, 16:29:34",318574,2023-10-17 21:44:30.900,2023-10-21 14:14:04.483,318573.583,BUT,BUT1,18.0,OBS
2,PSD_BUT_BUT1_PDM18_OBS,2023-10-17 21:44:30.900,BUT,18.0,OBS,2023-10-21 12:08:57.967,"3 dias, 14:24:27",311067,2023-10-17 21:44:30.900,2023-10-21 12:08:57.967,311067.067,BUT,BUT1,18.0,OBS
3,PSD_BUT_BUT1_PDM18_OBS,2023-10-17 21:44:30.900,BUT,18.0,OBS,2023-10-21 11:39:45.640,"3 dias, 13:55:15",309315,2023-10-17 21:44:30.900,2023-10-21 11:39:45.640,309314.740,BUT,BUT1,18.0,OBS
4,PSD_P_BUT1_PDM18_CA,2023-10-17 21:44:30.900,BUT,18.0,CA,2023-10-21 11:39:45.640,"3 dias, 13:55:15",309315,2023-10-17 21:44:30.900,2023-10-21 11:39:45.640,309314.740,BUT,BUT1,18.0,CA


In [98]:
# 1. Calcula a duração em segundos (caso ainda não tenha essa coluna)
alarmes["duracao_segundos"] = (
    alarmes["dt_normalizacao"] - alarmes["dt_alarme"]
).dt.total_seconds()

# 2. Identifica os alarmes de OBS com menos de 30s
obs_curto = alarmes["Alarme"].astype(str).str.contains(
    "OBS", case=False, na=False
) & (alarmes["duracao_segundos"] < 30)

# 3. Mantém apenas os alarmes válidos na tabela
alarmes = alarmes[~obs_curto].reset_index(drop=True)

In [99]:
# Ordena os DataFrames por estação e tempo
zm01 = zm01.sort_values(["estacao", "T"]).reset_index(drop=True)
alarmes = (
    alarmes.sort_values(["Estação", "dt_alarme"])
    .dropna(subset=["dt_alarme"])
    .reset_index(drop=True)
)

### Features

In [100]:
# Feature - Nome da estação
zm01['estacao'] = zm01['local_de_instalacao'].str.extract(r'PSD-([A-Z0-9]+)')

In [101]:
# Feature - Numero de Alarmes por Estação
zm01['numero_alarmes'] = zm01['estacao'].map(numeroAlarmesEstacao).fillna(0).astype(int)

In [102]:
# Feature - Alarmes Antes Manutencao

datas_alarmes_ordenadas = (
    alarmes["Horário do Alarme"].dropna().sort_values().values
)

total_alarmes = len(datas_alarmes_ordenadas)

zm01["alarmes_antes_manu"] = np.searchsorted(
    datas_alarmes_ordenadas, zm01["fim_avaria_datetime"].values, side="left"
)
zm01.loc[zm01["fim_avaria_datetime"].isna(), "alarmes_antes_manu"] = 0

In [103]:
# 1. Feature: dias_desde_ultima_manutencao
zm01["ultima_manu_fim"] = zm01.groupby("estacao")["fim_avaria_datetime"].shift(1)
zm01["dias_desde_ultima_manutencao"] = (zm01["T"] - zm01["ultima_manu_fim"]).dt.total_seconds() / 86400.0
zm01["dias_desde_ultima_manutencao"] = zm01["dias_desde_ultima_manutencao"].fillna(-1)

In [104]:
def calcular_janelas(df_zm, df_alt):
    cnt_24h = np.zeros(len(df_zm), dtype=int)
    cnt_7d = np.zeros(len(df_zm), dtype=int)

    for estacao, grp_zm in df_zm.groupby("estacao"):
        grp_alt = df_alt[df_alt["Estação"] == estacao]
        if grp_alt.empty:
            continue

        # Converte explicitamente para array nativo datetime64[ns] do NumPy (Evita o TypeError)
        times_alt = grp_alt["dt_alarme"].to_numpy(dtype="datetime64[ns]")
        times_zm = grp_zm["T"].to_numpy(dtype="datetime64[ns]")

        idx_t = np.searchsorted(times_alt, times_zm, side="right")
        idx_24h = np.searchsorted(
            times_alt, times_zm - np.timedelta64(24, "h"), side="left"
        )
        idx_7d = np.searchsorted(
            times_alt, times_zm - np.timedelta64(7, "D"), side="left"
        )

        cnt_24h[grp_zm.index] = idx_t - idx_24h
        cnt_7d[grp_zm.index] = idx_t - idx_7d

    return cnt_24h, cnt_7d

In [105]:
zm01["alarmes_ultimas_24h"], zm01["alarmes_ultimos_7d"] = calcular_janelas(zm01, alarmes)

media_diaria_7d = zm01["alarmes_ultimos_7d"] / 7.0
zm01["tendencia_alarmes"] = np.where(media_diaria_7d > 0, zm01["alarmes_ultimas_24h"] / media_diaria_7d, 0.0)

zm01['tendencia_alarmes'].value_counts()

tendencia_alarmes
0.000000    5069
7.000000      67
1.995098      23
0.052994      23
0.921374      22
            ... 
0.478632       1
0.663793       1
4.693798       1
4.569444       1
1.857516       1
Name: count, Length: 4551, dtype: int64

## One Hot Encoding

In [10]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42)

NameError: name 'train_test_split' is not defined